In [ ]:
from pathlib import Path
import os
import pandas as pd


try:
    PROJECT_DIR = Path(__file__).resolve().parents[0]
except NameError:
    PROJECT_DIR = Path(os.getcwd()).resolve().parents[0]
    
DATA_PATH = PROJECT_DIR / "data" / "online_retail.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Removing Duplicates

In [ ]:
df = pd.read_csv(DATA_PATH)
original_rows = len(df)
print(f"Original records: {original_rows}")

duplicates_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f"Number of duplicates removed: {duplicates_count}")
print(f"Records after removing duplicates: {len(df):,}")

Original records: 541909
Number of duplicates removed: 5268
Records after removing duplicates: 536,641


Convert data ypes

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"],errors="coerce")
df["CustomerID"] = (df["CustomerID"].astype("Int64").astype("string"))
df["Description"] = df["Description"].fillna("Unkown product")
for column in ["InvoiceNo","StockCode","Country"]:
    df[column] = df[column].str.strip()

In [ ]:
df["IsCancelled"] = df["InvoiceNo"].str.startswith("c", na=False)
df["IsNegativeQuantity"] = df["Quantity"] < 0
df["IsInvalidPrice"] = df["UnitPrice"] <= 0
df["IsMissingCustomer"] = df["CustomerID"].isna()
df["LineValue"] = df["Quantity"] * df["UnitPrice"]

df["IsValidSale"] = (
    ~df["isCancelled"]
    & (df["Quantity"] >0)
    & (df["UnitPrice"] > 0)
    & df["InvoiceDate"].notna()
)
  
     
transactions_path = OUTPUT_DIR/"transactions_clean.csv"
df.to_csv(transactions_path,index=False) 

sales = df.loc[df["IsValidSale"]].copy()
sales_path = OUTPUT_DIR/"sales_data.csv"
sales.to_csv(sales_path,index=False)  